In [1]:
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cpu
!pip install transformers librosa mamkit

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.1.2%2Bcpu-cp311-cp311-linux_x86_64.whl (184.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamkit 0.1.1 requires torch==2.3.0, but you have torch 2.1.2+cpu which is incompatible.
mamkit 0.1.1 requires torchaudio==2.3.0, but you have torchaudio 2.1.2+cpu which is incompatible.
torchtext 0.18.0 requires torch>=2.3.0, but you have torch 2.1.2+cpu which is incompatible.
  Using cached torch-2.3.0-cp311-cp311-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchaudio-2.3.0-cp311-cp311-manylinux1_x86_64.whl.metadata (6.4 kB)
Using cached torch-2.3.0-cp311-cp311-manylinux1_x86_64.whl (779.2 MB)
Using cached torchaudio-2.3.0-cp311-cp311-manylinux1_x86_64.whl (3.4 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchau

In [2]:
from pathlib import Path
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, WavLMModel
from mamkit.data.datasets import MMUSEDFallacy, InputMode
import torchaudio
from sklearn.metrics import classification_report, accuracy_score

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:484: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:341: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:341: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.11/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
class MultimodalDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128, audio_max_len=16000):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.audio_max_len = audio_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, audio_path_list, label, *_ = self.data[idx]
        audio_path = audio_path_list[0] if isinstance(audio_path_list, list) else audio_path_list

        encoding = self.tokenizer(text, padding='max_length', truncation=True,
                                  max_length=self.max_length, return_tensors='pt')

        waveform, sample_rate = torchaudio.load(audio_path)
        if sample_rate != 16000:
            waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        waveform = waveform.squeeze()
        waveform = F.pad(waveform, (0, max(0, self.audio_max_len - waveform.size(0))))[:self.audio_max_len]

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'audio_input': waveform,
            'label': torch.tensor(label)
        }


In [5]:
# --- 3. Model Class ---
class MultimodalClassifier(nn.Module):
    def __init__(self, num_labels):
        super(MultimodalClassifier, self).__init__()
        self.text_encoder = RobertaModel.from_pretrained("/content/drive/MyDrive/models/roberta-base")
        self.audio_encoder = WavLMModel.from_pretrained("/content/drive/MyDrive/models/wavlm-base-plus")
        self.dropout = nn.Dropout(0.3)
        self.hidden = nn.Linear(self.text_encoder.config.hidden_size + self.audio_encoder.config.hidden_size, 256)
        self.norm = nn.LayerNorm(256)
        self.classifier = nn.Linear(256, num_labels)

    def forward(self, input_ids, attention_mask, audio_input):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        audio_outputs = self.audio_encoder(audio_input)

        text_pooled = text_outputs.pooler_output
        audio_pooled = audio_outputs.last_hidden_state.mean(dim=1)

        combined = torch.cat((text_pooled, audio_pooled), dim=1)
        hidden = self.norm(self.hidden(self.dropout(combined)))
        logits = self.classifier(hidden)
        return logits



In [6]:
# --- 4. Load Tokenizer ---
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:


# --- 5. Load Test Data for Task A and Task B ---
DATA_PATH = Path("/content/base")

# Task A: Fallacy Detection (Binary)
loader_a = MMUSEDFallacy(task_name="afd", input_mode=InputMode.TEXT_AUDIO, base_data_path=DATA_PATH)
splits_a = list(loader_a.get_splits("mancini-et-al-2024"))
test_dataset_a = MultimodalDataset(splits_a[0].test, tokenizer)
test_loader_a = DataLoader(test_dataset_a, batch_size=8, shuffle=False)

# Task B: Fallacy Classification (6 classes)
loader_b = MMUSEDFallacy(task_name="afc", input_mode=InputMode.TEXT_AUDIO, base_data_path=DATA_PATH)
splits_b = list(loader_b.get_splits("mancini-et-al-2024"))
test_dataset_b = MultimodalDataset(splits_b[0].test, tokenizer)
test_loader_b = DataLoader(test_dataset_b, batch_size=8, shuffle=False)


Building AFC Context: 100%|██████████| 1278/1278 [00:00<00:00, 8112.82it/s]


In [10]:
# --- 6. Load Models ---
model_a = MultimodalClassifier(num_labels=2).to(device)
model_a.load_state_dict(torch.load("/content/drive/MyDrive/task_a_model.pt", map_location=torch.device('cpu')))
model_a.to(device)
model_a.eval()

model_b = MultimodalClassifier(num_labels=6).to(device)
model_b.load_state_dict(torch.load("/content/drive/MyDrive/task_b_model.pt", map_location=torch.device('cpu')))
model_b.to(device)
model_b.eval()


MultimodalClassifier(
  (text_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [11]:
# --- 7. Inference Function ---
def evaluate_model(model, dataloader):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            audio_input = batch['audio_input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask, audio_input)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds


In [12]:
# --- 8. Run Evaluation ---

# Task A
labels_a, preds_a = evaluate_model(model_a, test_loader_a)
print("== Task A: Fallacy Detection ==")
print(classification_report(labels_a, preds_a, digits=4))
acc_a = accuracy_score(labels_a, preds_a)

# Task B
labels_b, preds_b = evaluate_model(model_b, test_loader_b)
print("== Task B: Fallacy Classification ==")
print(classification_report(labels_b, preds_b, digits=4))
acc_b = accuracy_score(labels_b, preds_b)




== Task A: Fallacy Detection ==
              precision    recall  f1-score   support

           0     0.9695    0.9252    0.9468       481
           1     0.4627    0.6889    0.5536        45

    accuracy                         0.9049       526
   macro avg     0.7161    0.8070    0.7502       526
weighted avg     0.9261    0.9049    0.9132       526

== Task B: Fallacy Classification ==
              precision    recall  f1-score   support

           0     0.9545    0.6562    0.7778        32
           1     0.2500    0.5000    0.3333         2
           2     0.0000    0.0000    0.0000         0
           3     0.2000    1.0000    0.3333         1
           4     0.5000    0.5000    0.5000         2
           5     0.5000    1.0000    0.6667         1

    accuracy                         0.6579        38
   macro avg     0.4008    0.6094    0.4352        38
weighted avg     0.8617    0.6579    0.7251        38



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
from sklearn.metrics import f1_score

# --- 9. Combined Accuracy and F1 Score ---
combined_acc = (acc_a * len(labels_a) + acc_b * len(labels_b)) / (len(labels_a) + len(labels_b))

# Compute macro F1 scores
f1_a = f1_score(labels_a, preds_a, average='macro')
f1_b = f1_score(labels_b, preds_b, average='macro')

# Weighted macro F1 based on dataset sizes
combined_f1 = (f1_a * len(labels_a) + f1_b * len(labels_b)) / (len(labels_a) + len(labels_b))

print(f"\n Combined Accuracy of Model Across Both Tasks: {combined_acc:.4f}")
print(f" Combined Macro F1 Score Across Both Tasks: {combined_f1:.4f}")



 Combined Accuracy of Model Across Both Tasks: 0.8883
 Combined Macro F1 Score Across Both Tasks: 0.7290
